# 10 - Data Cleaning & Text Preprocessing (Production Dataset)
## ShopEase Europe | Sentiment Analysis Project - Phase 2 
**Objective:** Clean the production dataset, engineer temporal and 
text-based features, and preprocess review text into a model-ready 
format for both classical and transformer-based modelling.

## Import Libaries

In [1]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

Libraries loaded successfully


## Load the Dataset

In [2]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'amazon_reviews_cleaned.csv')
PROCESSED_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'production_clean_reviews.csv')

df = pd.read_csv(RAW_DATA_PATH)

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Dataset loaded: 21,055 rows x 7 columns


## Structural Cleaning
Standardising sentiment casing, handling the missing country value, 
and converting timestamp to a proper datetime format.

In [4]:
# Standardise sentiment to lowercase
df['sentiment'] = df['sentiment'].str.lower()

# Handle missing country
print(f"Missing country values before: {df['country'].isnull().sum()}")
df['country'] = df['country'].fillna('Unknown')
print(f"Missing country values after: {df['country'].isnull().sum()}")

# Convert timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed')

print(f"\nSentiment values: {df['sentiment'].unique()}")
print(f"Timestamp dtype: {df['timestamp'].dtype}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

Missing country values before: 0
Missing country values after: 0

Sentiment values: ['negative' 'positive' 'neutral']
Timestamp dtype: datetime64[ns, UTC]
Date range: 2007-08-27 17:25:01+00:00 to 2024-09-17 13:19:27+00:00


## Structural Cleaning Finding

The dataset spans 17 years, from August 2007 to September 2024. This 
wide range likely reflects genuine historical Amazon review data 
spanning many years rather than data generated for a specific recent 
campaign or product launch period.

One missing country value was identified and filled as "Unknown" 
rather than dropped, preserving the review for sentiment and text 
analysis while excluding it from country-specific breakdowns.

## Text Cleaning

In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[?]{2,}', '?', text)
    text = re.sub(r'[.]{2,}', '.', text)
    text = re.sub(r'[^a-zA-Z0-9\s\.,!?\'-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_review'] = df['review'].apply(clean_text)

print(f"Sample original:\n{df['review'].iloc[0][:200]}")
print(f"\nSample cleaned:\n{df['cleaned_review'].iloc[0][:200]}")
print(f"\nOriginal length avg:  {df['review'].str.len().mean():.1f} characters")
print(f"Cleaned length avg:   {df['cleaned_review'].str.len().mean():.1f} characters")

Sample original:
I registered on the website, tried to order a laptop, entered all the details, but instead of charging me and sending the product, they froze my account, demanding various verification documents. I se

Sample cleaned:
i registered on the website, tried to order a laptop, entered all the details, but instead of charging me and sending the product, they froze my account, demanding various verification documents. i se

Original length avg:  460.8 characters
Cleaned length avg:   457.3 characters


## Text Cleaning Finding

Average review length dropped only slightly from 460.8 to 457.3 
characters after cleaning, indicating minimal noise such as URLs, 
excessive punctuation, or special characters in the original text. 
This is expected for genuine, manually written reviews compared to 
more heavily formatted or templated text.